# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, leveraging Linked Data capabilities and referring to dataset elements via their `@id` for reproducibility.

### Dataset Source
The dataset is described by a Croissant JSON-LD schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the records in the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_dict = dataset.metadata.to_json()

print('Dataset Name:', dataset.metadata.name)
print('Identifier:', getattr(dataset.metadata, 'identifier', 'N/A'))
print('License:', getattr(dataset.metadata, 'license', 'N/A'))
print('Description:', dataset.metadata.description)
print('Published:', getattr(dataset.metadata, 'datePublished', 'N/A'))

## 2. Croissant Structure Overview

Let's list the available record sets (tables), their `@id`s, and their fields, as available in the Croissant schema.

All references to entities (record sets, fields) will use their Croissant `@id` for clarity and reproducibility.

In [ ]:
# List all record sets and their fields by @id

record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record sets in the dataset.\n")
all_record_set_ids = []  # will be used later

for record_set in record_sets:
    print(f"Record Set Name: {getattr(record_set, 'name', '')}")
    print(f"  @id: {getattr(record_set, '@id', getattr(record_set, 'id', ''))}")
    all_record_set_ids.append(getattr(record_set, '@id', getattr(record_set, 'id', '')))
    print("  Fields:")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"    - {getattr(field, 'name', '')} (@id: {getattr(field, '@id', getattr(field, 'id', ''))}, type: {getattr(field, 'data_type', getattr(field, 'type', ''))})")
    print("")

## 3. Data Extraction

We can now extract the data from the record sets we listed above into pandas DataFrames, referencing each by its Croissant `@id`.

In [ ]:
# Convert all record sets to pandas DataFrames
dataframes = {}

for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set {rs_id} with shape {df.shape}")

# Let's inspect the columns of the largest (first) record set
main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None
if main_record_set_id:
    print(f"Columns in record set ({main_record_set_id}):")
    print(list(dataframes[main_record_set_id].columns))
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll select a quantitative or numeric field for analysis, such as `MW (kDa)` (molecular weight) or `Coverage (%)`. We'll reference the field using its Croissant `@id` as seen in the earlier overview.

In [ ]:
# For demonstration, choose a field likely to be numeric - update field_id if your dataset differs
main_df = dataframes[main_record_set_id]

# Find likely numeric fields by scanning columns
numeric_like = [col for col in main_df.columns if main_df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(main_df[col])]
if not numeric_like:
    # Try to convert columns to numeric where possible
    possible_numeric_cols = []
    for col in main_df.columns:
        try:
            coerced = pd.to_numeric(main_df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                possible_numeric_cols.append(col)
        except Exception:
            pass
    numeric_like = possible_numeric_cols

print(f"Numeric-like fields identified: {numeric_like}\n")
# We'll pick the first available numeric field (e.g., 'MW (kDa)' or 'Coverage (%)')
numeric_field = numeric_like[0] if numeric_like else None
group_field = None
# Suggest a possible group field (e.g., based on description, Accession, etc.)
for col in main_df.columns:
    if ('group' in col.lower()) or ('description' in col.lower()) or ('id' in col.lower()):
        group_field = col
        break

if numeric_field:
    # Coerce to float just in case
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
    threshold = main_df[numeric_field].mean()
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}):\n", filtered_df[[numeric_field]].head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group if possible
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field}:\n", grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization

Let's plot the distribution of the selected numeric field, and (if grouped field exists) compare means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If we have a grouping field, plot group means
    if group_field and group_field in main_df.columns:
        group_means = main_df.dropna(subset=[numeric_field, group_field]).groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(8, 5))
        sns.barplot(x=group_means.values, y=group_means.index, orient='h')
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(f"{group_field}")
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric fields identified for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR² dataset schema and records using `mlcroissant`, referencing all data elements by their Croissant `@id`s for reproducibility.
- Inspected all record sets and their schema via JSON-LD structure.
- Loaded the main data table and identified quantitative and categorical fields for exploratory analysis.
- Filtered, normalized, and grouped data for meaningful summaries.
- Visualized key numeric distributions from the proteomics dataset.

This approach demonstrates FAIR and interoperable analysis by leveraging Croissant metadata and record structure, suitable for scaling to larger linked datasets in the future.